# RETINA-ND — pre-submission code/artifact test

Run this notebook first against the completed project folder. It does **not**
rerun differential expression. It checks the release inputs, critical frozen
outputs, final manuscript-facing counts, and final figures before you run the
full clean notebook from scratch.


In [ ]:
# ============================================================
# 00 — RELEASE SETUP
# ============================================================

import os
import sys
import json
import platform
import subprocess
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone

# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    mount = Path("/content/drive")
    if not (mount / "MyDrive").exists():
        drive.mount(str(mount), force_remount=False)
except Exception:
    pass

# Prefer the user's completed project when it exists; otherwise use the
# standard release folder name. The environment variable can override both.
candidate_existing = Path("/content/drive/MyDrive/RETINA_ND_V2 (1)")
candidate_standard = Path("/content/drive/MyDrive/RETINA_ND_V2")

if "RETINA_ND_ROOT" in os.environ:
    PROJECT_ROOT = Path(os.environ["RETINA_ND_ROOT"])
elif candidate_existing.exists():
    PROJECT_ROOT = candidate_existing
else:
    PROJECT_ROOT = candidate_standard

PROJECT_ROOT = PROJECT_ROOT.resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["RETINA_ND_ROOT"] = str(PROJECT_ROOT)

ROOT = PROJECT_ROOT
LOCKS = ROOT / "locks"
MANIFESTS = ROOT / "manifests"
RESULTS = ROOT / "results"

for p in [
    LOCKS,
    MANIFESTS,
    RESULTS,
    ROOT / "downloads",
    ROOT / "scripts",
    ROOT / "logs",
    ROOT / "resources",
]:
    p.mkdir(parents=True, exist_ok=True)

STRICT_REFERENCE_CHECKS = True
PROJECT_VERSION = "2.0-biorxiv-release"

print("RETINA-ND project root:", ROOT)
print("UTC:", datetime.now(timezone.utc).isoformat())
print("Python:", platform.python_version())
print("STRICT_REFERENCE_CHECKS:", STRICT_REFERENCE_CHECKS)


In [ ]:
# ============================================================
# QUICK PRE-SUBMISSION CHECK
# ============================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path(__import__("os").environ["RETINA_ND_ROOT"])

required_inputs = [
    ROOT / "resources" / "pathways" / "ReactomePathways_V97.gmt",
    ROOT / "ocular_validation" / "Wolf_Cell_2023" / "NIHMS1931846-supplement-8.xlsx",
    ROOT / "ocular_validation" / "PXD073336" / "Retinaxhippocampus_Proteomics.xlsx",
]

required_outputs = [
    ROOT / "results" / "cross_disease_convergence" / "AD_PD_REACTOME_V97_PATHWAY_CONVERGENCE_v1.csv.gz",
    ROOT / "results" / "final_evidence" / "RETINA_ND_CORE_MULTIMODAL_EVIDENCE_MATRIX_v3.csv.gz",
    ROOT / "results" / "final_evidence" / "RETINA_ND_UNIQUE_GENE_CANDIDATE_RANKING_v4.csv",
]

print("INPUT CHECK")
for p in required_inputs:
    print(" ", "OK" if p.exists() else "MISSING", p)

print("\nOUTPUT CHECK")
for p in required_outputs:
    print(" ", "OK" if p.exists() else "MISSING", p)

missing = [p for p in required_inputs + required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Pre-submission check failed. Missing:\n" +
        "\n".join(str(p) for p in missing)
    )

pathway = pd.read_csv(required_outputs[0], compression="gzip")
ranking = pd.read_csv(required_outputs[2])

r3_n = pathway.loc[pathway["pathway_priority"].eq("R3_recurrent_directional"), "pathway"].nunique()
assert r3_n == 665, f"R3 count mismatch: {r3_n}"

top8 = ranking.head(8)["gene_symbol"].astype(str).tolist()
expected_top8 = ["SCAMP5", "XYLB", "RABEPK", "ITGAM", "CD38", "CTCF", "UBTD2", "NAPEPLD"]
assert top8 == expected_top8, f"Top-8 mismatch: {top8}"

print("\nCore frozen checks: PASS")
print("R3 pathways:", r3_n)
print("Top 8:", top8)
